In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import json
import numpy as np
import time
import pandas as pd

from src.evaluation import *
from src.bedrock import get_bearer_token

import warnings
warnings.filterwarnings('ignore')

In [29]:
get_bearer_token()

MAX_ERRORS = 10
OPENSEARCH_DOMAIN_FQDN = 'https://vpc-int-use1-opensearch-ml-c32nkeiaudrckcr2ep7jghefje.us-east-1.es.amazonaws.com'
HEADERS = {
    'Content-Type': 'application/json',
    'Accept-Encoding': 'gzip',
}

# INDEX_NAME = 'qna-data'
INDEX_NAME = 'search-data-titan-embed-2'

query_json_path = "data/variantQna.json"
with open( query_json_path, "r" ) as file:
    test_query_data_variant = json.load(file)

query_json_path = "data/final-questions.json"
with open( query_json_path, "r" ) as file:
    test_query_data_final = json.load(file)

Bearer token set as BEARER_TOKEN_STR global variable


In [4]:
len(test_query_data_variant)

2307

In [5]:
len(test_query_data_final)

1841

In [6]:
diff = [item for item in test_query_data_variant if item not in test_query_data_final]
len(diff)

484

In [7]:
import random

random.seed(0)

In [8]:
# sampling
# final_sample = random.sample(test_query_data_final, 100)
# diff_sample = random.sample(diff, 100)

In [9]:
rerank_prompt_claude = """
<task>
Identify the most relevant search result that best matches the intent of the query.
</task>

<instructions>
1. Exact Match Analysis (Highest Priority):
- Check if the query appears word-for-word in the result's example questions
- Match between query keywords and the result's primary topic description
- Look for exact matches in the intent description

2. Topic Focus:
- Whether the result is dedicated to answering this specific type of question
- Whether the query topic is the main focus vs. being a secondary topic
- How directly the result addresses the query subject

3. Example Questions Alignment:
- How closely the example questions match the query pattern
- Whether the example questions cover the same information type
- Whether the examples suggest the result can provide the specific information needed

4.You must return the index of the most relevant search result from 0 to {max_idx}.
</instructions>

<query>
{query}
</query>

<search_results>
{formatted_results}
</search_results>

<output_format>
Return only the index (from 0 to {max_idx}) of the best result.
Example: 2
</output_format>

Think step-by-step before returning ONLY the index without any explanation:
"""

In [10]:
rerank_prompt_nova = """
##task##
Identify the most relevant search result that best matches the intent of the query.

##instructions##
1. Exact Match Analysis (Highest Priority):
- Check if the query appears word-for-word in the result's example questions
- Match between query keywords and the result's primary topic description
- Look for exact matches in the intent description

2. Topic Focus:
- Whether the result is dedicated to answering this specific type of question
- Whether the query topic is the main focus vs. being a secondary topic
- How directly the result addresses the query subject

3. Example Questions Alignment:
- How closely the example questions match the query pattern
- Whether the example questions cover the same information type
- Whether the examples suggest the result can provide the specific information needed

4.You must return the index of the most relevant search result from 0 to {max_idx}.

##query##
{query}

##search_results##
{formatted_results}

##output_format##
Return ONLY the index (from 0 to {max_idx}) of the best result.
Example: 2

Think step-by-step before returning ONLY the index without any new lines, explanation, or additional text:
"""

In [11]:
models = ["sonnet35_v2", "nova_pro", "nova_lite"]
eval_results = {}

In [12]:
from tqdm.auto import tqdm

In [18]:
for model in models:
    if model == "sonnet35_v2":
        results_dfs = evaluate_queries(
            query_data=test_query_data_final,
            opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
            index_name=INDEX_NAME,
            headers=HEADERS,
            top_k=5,
            embed_model="amazon.titan-embed-text-v1-pgo",
            rerank_model=model,
            rerank_prompt=rerank_prompt_claude)
    else:
        results_dfs = evaluate_queries(
            query_data=test_query_data_final,
            opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
            index_name=INDEX_NAME,
            headers=HEADERS,
            top_k=5,
            embed_model="amazon.titan-embed-text-v1-pgo",
            rerank_model=model,
            rerank_prompt=rerank_prompt_nova)

    summary_metrics = {}
    for search_type, df in results_dfs.items():
        summary_metrics[search_type] = get_summary_metrics(df)

    combined_metrics = pd.concat(summary_metrics, axis=1)
    combined_metrics.columns = combined_metrics.columns.get_level_values(0)
    eval_results[model] = combined_metrics

  0%|          | 0/1841 [00:00<?, ?it/s]

Bearer token set as BEARER_TOKEN_STR global variable
Bearer token set as BEARER_TOKEN_STR global variable
Bearer token set as BEARER_TOKEN_STR global variable


In [19]:
eval_results["sonnet35_v2"]

,hybrid_boost_0.0001,hybrid_boost_0.0001_reranked
Accuracy,0.756111,0.807170
Avg Latency,0.662601,2.270663
P10 Latency,0.439013,1.814853
P90 Latency,0.824187,2.680650
P99 Latency,1.559557,3.324314
Max Latency,9.248769,23.973777


In [20]:
eval_results["nova_pro"]

,hybrid_boost_0.0001,hybrid_boost_0.0001_reranked
Accuracy,0.756111,0.806627
Avg Latency,0.666585,1.275832
P10 Latency,0.435443,1.019904
P90 Latency,0.825779,1.495574
P99 Latency,1.265567,2.007988
Max Latency,15.424008,15.917753


In [21]:
eval_results["nova_lite"]

,hybrid_boost_0.0001,hybrid_boost_0.0001_reranked
Accuracy,0.756111,0.814231
Avg Latency,0.683585,1.282410
P10 Latency,0.434322,1.015830
P90 Latency,0.817498,1.458778
P99 Latency,1.259172,2.163146
Max Latency,57.106345,57.649274


In [22]:
for k, v in eval_results.items():
    eval_results[k] = v.add_suffix(f"_{k}")
merged_df = pd.concat(list(eval_results.values()), axis=1)
merged_df

,hybrid_boost_0.0001_nova_pro,hybrid_boost_0.0001_reranked_nova_pro,hybrid_boost_0.0001_nova_lite,hybrid_boost_0.0001_reranked_nova_lite,hybrid_boost_0.0001_sonnet35_v2,hybrid_boost_0.0001_reranked_sonnet35_v2
Accuracy,0.756111,0.806627,0.756111,0.814231,0.756111,0.807170
Avg Latency,0.666585,1.275832,0.683585,1.282410,0.662601,2.270663
P10 Latency,0.435443,1.019904,0.434322,1.015830,0.439013,1.814853
P90 Latency,0.825779,1.495574,0.817498,1.458778,0.824187,2.680650
P99 Latency,1.265567,2.007988,1.259172,2.163146,1.559557,3.324314
Max Latency,15.424008,15.917753,57.106345,57.649274,9.248769,23.973777


In [23]:
merged_df.to_csv("evaluation_results_final_sample_v2_complete.csv")

In [24]:
eval_results_diff = {}

In [30]:
for model in models:
    if model == "sonnet35_v2":
        results_dfs = evaluate_queries(
            query_data=diff,
            opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
            index_name=INDEX_NAME,
            headers=HEADERS,
            top_k=5,
            embed_model="amazon.titan-embed-text-v1-pgo",
            rerank_model=model,
            rerank_prompt=rerank_prompt_claude)
    else:
        results_dfs = evaluate_queries(
            query_data=diff,
            opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
            index_name=INDEX_NAME,
            headers=HEADERS,
            top_k=5,
            embed_model="amazon.titan-embed-text-v1-pgo",
            rerank_model=model,
            rerank_prompt=rerank_prompt_nova)

    summary_metrics = {}
    for search_type, df in results_dfs.items():
        summary_metrics[search_type] = get_summary_metrics(df)

    combined_metrics = pd.concat(summary_metrics, axis=1)
    combined_metrics.columns = combined_metrics.columns.get_level_values(0)
    eval_results_diff[model] = combined_metrics

  0%|          | 0/484 [00:00<?, ?it/s]

In [31]:
for k, v in eval_results_diff.items():
    eval_results_diff[k] = v.add_suffix(f"_{k}")
merged_diff_df = pd.concat(list(eval_results_diff.values()), axis=1)
merged_diff_df

,hybrid_boost_0.0001_sonnet35_v2,hybrid_boost_0.0001_reranked_sonnet35_v2,hybrid_boost_0.0001_nova_pro,hybrid_boost_0.0001_reranked_nova_pro,hybrid_boost_0.0001_nova_lite,hybrid_boost_0.0001_reranked_nova_lite
Accuracy,0.778926,0.776860,0.778926,0.785124,0.778926,0.816116
Avg Latency,0.629130,2.029763,0.681251,1.272511,0.659624,1.286602
P10 Latency,0.433564,1.647139,0.437228,0.964696,0.443823,1.003546
P90 Latency,0.797658,2.480977,0.839932,1.505531,0.829860,1.574332
P99 Latency,0.918554,2.798278,1.540287,2.411549,1.039421,2.328589
Max Latency,3.166469,4.665767,6.914209,7.517272,3.708614,4.174753


In [32]:
merged_diff_df.to_csv("evaluation_results_final_sample_v2_complete_diff.csv")